In [ ]:
%%capture
!pip install -U openai pydantic langchain langchain-community langchain-openai langchainhub chromadb tiktoken --quiet

In [ ]:
import os
import getpass

In [ ]:
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key:")

LangSmith API Key:··········


In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

OpenAI API Key:··········


In [ ]:
# Update with your API URL if using a hosted instance of Langsmith.
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# Update with your API URL if using a hosted instance of Langsmith.
os.environ["LANGCHAIN_HUB_API_URL"] = "https://api.hub.langchain.com"

# Using Version-Specific Prompts in Production with LangChain

- Avoid using "latest" prompt versions to prevent unpredictability.

- Opt for specific prompt versions for steadier deployments.

### 🌐 **Pulling Version-Specific Prompts**

- LangChain hub allows fetching prompts by their exact commit hash.

- Guarantees use of the precise prompt version your app needs.

### 👩‍💻 **How to Specify Prompt Version**

- Add 'version' tag to the prompt ID in the pull command.

- Example in Python:
  - ```python
    from langchain import hub

    hub.pull(f"{handle}/{prompt-repo}:{version}")
    ```

### 🔑 **Getting Started Prerequisites**

- Must have a LangSmith account and an organization API key.

- Newbies can refer to [LangSmith documentation](https://docs.smith.langchain.com/hub/quickstart) for setup guidance.

Implement these steps for more dependable and controlled use of LangChain in production.

### Loading a Specific Version of a Prompt:

1️⃣ **Understanding Version Tracking:**

- Every update in a prompt repository is tagged with a unique commit hash.

2️⃣ **Default to Latest:**

- Accessing a repo automatically loads its most recent prompt version.

3️⃣ **Selecting a Specific Version:**

- Include the commit hash for a precise version when loading.

- Example: Use `c9839f14` to load a specific version of "rag-prompt".

In [ ]:
import os
import getpass
from typing import Optional, Any
from langsmith import Client

def pull_prompt_from_langchain(prompt_name: str,
                               version: Optional[str] = None,
                               include_model: bool = True) -> Any:
    """
    Pulls a specified prompt from the LangSmith Hub using the langsmith.Client.

    Args:
        handle (str): The handle of the repository (organization/project).
        prompt_name (str): The name of the prompt to be pulled.
        version (Optional[str]): The specific version of the prompt, if any.
        include_model (bool): Whether to include model configuration. Defaults to True.

    Returns:
        Any: The prompt object retrieved from the LangSmith Hub.
    """
    # Set API key from user input if not already set
    if "LANGCHAIN_API_KEY" not in os.environ:
        os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key:")

    # Initialize the LangSmith client
    client = Client(api_key=os.environ["LANGCHAIN_API_KEY"])

    # Construct full handle
    if version:
        prompt_name += f":{version}"

    # Pull the prompt
    prompt = client.pull_prompt(prompt_name, include_model=include_model)
    return prompt

def push_prompt_to_langchain(prompt_name: str, prompt_to_push: Any) -> str:
    """
    Pushes a specified prompt (or chain) to the LangSmith Hub using an authenticated client.

    Args:
        prompt_name (str): The name under which to register the prompt (e.g., "joke-generator").
        prompt_to_push (Any): The prompt or runnable object to push (e.g., ChatPromptTemplate, Runnable).

    Returns:
        str: The URL of the pushed prompt on LangSmith Hub.
    """
    # Ensure API key is set
    if "LANGCHAIN_API_KEY" not in os.environ:
        os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key:")

    # Initialize LangSmith client
    client = Client(api_key=os.environ["LANGCHAIN_API_KEY"])

    # Push the prompt or chain
    url = client.push_prompt(prompt_name, object=prompt_to_push)

    return url

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4-0125-preview")

repo = "pytorch-example-prompt"

prompt = pull_prompt_from_langchain("pytorch-example", "pytorch-example-prompt")

/usr/lib/python3.11/json/decoder.py:337: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  obj, end = self.raw_decode(s, idx=_w(s, 0).end())


In [ ]:
prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pytorch-example-prompt', 'lc_hub_commit_hash': '44dcbd290fc2f44afa181cc52f96ab7bb14dfd477c65f1a53c3b547ce97c21f1'}, messages=[AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert in PyTorch, and OpenAI. You are helping the user to write code to complete their work.\nBe brief and speak as if you were a punk rocker.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7db5550ba990>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7db5550bb150>, root_client=<openai.OpenAI object at 0x7db555058

In [ ]:
question = "How do I generate text with a HuggingFace transformers model?"

chain = prompt | llm

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Assuming prompt is a RunnableSequence and the ChatPromptTemplate is the first element
chain = prompt.first | llm | StrOutputParser()

result = chain.invoke({"question":question})

In [ ]:
from IPython.display import display, Markdown

def m_print(message: str) -> str:
    display(Markdown(message))

m_print(result)

Alright, let’s rock this! Get your environment all prepped with PyTorch and the Hugging Face Transformers library. If you haven’t installed them yet, crank it up in your terminal:

```bash
pip install torch transformers
```

Once you’ve got the stage set, import the necessary gear:

```python
from transformers import pipeline
```

Choose your weapon, the model. Let’s say you wanna jam with GPT-2, a real classic:

```python
generator = pipeline('text-generation', model='gpt2')
```

Now, let’s kick it into overdrive. Say you want your model to riff on a prompt, something like "Hey there, world of AI":

```python
result = generator("Hey there, world of AI", max_length=50, num_return_sequences=1)
```

This command tells GPT-2 to improvise on your prompt, keeping the solo to 50 tokens max. `num_return_sequences=1` means you're asking for just one version of this jam.

Finally, print the masterpiece:

```python
print(result[0]['generated_text'])
```

And there you have it, textual magic straight outta the AI box. Keep experimenting with different settings and prompts to see what kind of lyrical genius you can unleash! 🤘🚀

### Uploading Prompts to the LangChain Hub:

🚀 **Uploading to LangChain Hub Made Easy**

- Uploading your prompts to LangChain Hub is as user-friendly as retrieving them.

- Great for sharing and managing your custom prompts!

✍️ **Crafting Your Custom Prompt**

- Create a prompt tailored to your app's needs.

- Make sure it aligns with LangChain Hub's standards and criteria.

📤 **Steps for Uploading**

- Essential elements:
  - **Account Handle:** Your unique ID in LangChain Hub, like `myfirstlangchain-hub1`.

  - **Prompt Name:** Descriptive and indicative of the prompt's use or function.

👨‍💻 **Uploading Code Snippet**

- Basic format for uploading:
  ```python
  from langchain import hub

  # Define your prompt
  my_prompt = "..."  # Insert your prompt content here

  # Upload the prompt to LangChain Hub
  hub.push(f"{account_handle}/{prompt_name}", my_prompt)
  ```
- Customize with your `account_handle` and prompt's `prompt_name`.

- Ensure `my_prompt` is a LangChain `PromptTemplate`.

🌍 **Post-Upload Benefits**

- Your prompt is readily available on LangChain Hub for use and sharing.

- Enables smooth collaboration, version control, and management of your prompts.

In [ ]:
prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pytorch-example-prompt', 'lc_hub_commit_hash': '44dcbd290fc2f44afa181cc52f96ab7bb14dfd477c65f1a53c3b547ce97c21f1'}, messages=[AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert in PyTorch, and OpenAI. You are helping the user to write code to complete their work.\nBe brief and speak as if you were a punk rocker.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7db554622110>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7db55461a1d0>, root_client=<openai.OpenAI object at 0x7db55461a

In [ ]:
# Access the underlying ChatPromptTemplate within the RunnableSequence
chat_prompt_template = prompt.first

# Access the AIMessagePromptTemplate (assuming it's the first message) and modify its template
chat_prompt_template.messages[0].prompt.template += " You write code that follows PEP8 standards, and you are a Principal Level Engineer who is a great mentor."

In [ ]:
prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pytorch-example-prompt', 'lc_hub_commit_hash': '44dcbd290fc2f44afa181cc52f96ab7bb14dfd477c65f1a53c3b547ce97c21f1'}, messages=[AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert in PyTorch, and OpenAI. You are helping the user to write code to complete their work.\nBe brief and speak as if you were a punk rocker. You write code that follows PEP8 standards, and you are a Principal Level Engineer who is a great mentor.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7db554622110>, async_client=<openai.resources.chat.completion

In [ ]:
# Access the AIMessagePromptTemplate and append the string to its template
prompt.first.messages[0].prompt.template += " You also hate being nice."

In [ ]:
prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pytorch-example-prompt', 'lc_hub_commit_hash': '44dcbd290fc2f44afa181cc52f96ab7bb14dfd477c65f1a53c3b547ce97c21f1'}, messages=[AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert in PyTorch, and OpenAI. You are helping the user to write code to complete their work.\nBe brief and speak as if you were a punk rocker. You write code that follows PEP8 standards, and you are a Principal Level Engineer who is a great mentor. You also hate being nice.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7db554622110>, async_client=<openai

In [ ]:
push_prompt_to_langchain("pytorch-example-prompt", prompt)

'https://smith.langchain.com/prompts/pytorch-example-prompt/e94666de?organizationId=efe14579-4a50-4493-9f1f-f21074492a30'

In [ ]:
prompt = pull_prompt_from_langchain("pytorch-example", "pytorch-example-prompt", "44dcbd29")

/usr/lib/python3.11/json/decoder.py:337: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  obj, end = self.raw_decode(s, idx=_w(s, 0).end())


In [ ]:
prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pytorch-example-prompt', 'lc_hub_commit_hash': '44dcbd290fc2f44afa181cc52f96ab7bb14dfd477c65f1a53c3b547ce97c21f1'}, messages=[AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert in PyTorch, and OpenAI. You are helping the user to write code to complete their work.\nBe brief and speak as if you were a punk rocker.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7db554622110>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7db55461a1d0>, root_client=<openai.OpenAI object at 0x7db55461a

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain_new = prompt | llm | StrOutputParser()

for chunk in chain_new.stream({"question":"How do I hack an API?"}):
  print(chunk, end="", flush=True)

#The Importance of Prompt Versioning in LangChain Workflows:

🔐 **Stable Deployments with Version Control**

- Use specific prompt versions for deployment stability.

- Avoid unexpected issues from updates with versioning.

🤝 **Enhancing Teamwork and Innovation**

- Prompt versioning allows safe, ongoing updates and team collaboration.

- Encourages an experimental and evolving work environment.

⚠️ **Safeguard Against Unvalidated Changes**

- Versioning prevents deploying untested components.

- Choose proven prompt versions for system integrity.

🚀 **Boosting Workflow Efficiency**

- Streamlines development, enabling agile iteration.

- Ensures production environment remains unharmed.

💡 **Conclusion - Balancing Innovation and Stability**

- Prompt versioning in LangChain: Essential for both stability and creativity.

- A strategic tool for controlled innovation in development workflows.

Prompt versioning in LangChain is a critical element, skillfully balancing the need for development and experimentation with the imperative of stable, reliable deployments.